In [2]:
import os
import numpy as np
import cv2
from PIL import Image, ImageStat
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
from collections import Counter

def estimate_jpeg_quality(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, laplacian = cv2.threshold(cv2.convertScaleAbs(cv2.Laplacian(gray, 3)), 0, 255, cv2.THRESH_BINARY)
    return np.mean(laplacian)

def extract_features(image_path):
    try:
        img = Image.open(image_path)
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            print(f"Warning: Unable to read {image_path} with OpenCV. Skipping...")
            return None
    except:
        print(f"Error: Unable to open {image_path}. Skipping...")
        return None
    
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    
    features = []
    
    jpeg_quality = estimate_jpeg_quality(img_cv)
    features.append(jpeg_quality)
    
    sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()
    features.append(sharpness)
    
    stat = ImageStat.Stat(img)
    for channel in range(3):
        features.extend([stat.mean[channel], stat.rms[channel], stat.var[channel]])
    
    hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
    hist = hist / hist.sum()
    entropy = -np.sum(hist * np.log2(hist + 1e-10))
    features.append(entropy)
    
    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    edge_density = np.mean(np.sqrt(sobel_x**2 + sobel_y**2))
    features.append(edge_density)
    
    for i in range(3):
        channel = img_cv[:,:,i]
        features.append(np.max(channel) - np.min(channel))
    
    return features

real_path = "C://Users//Sinchan A//Desktop//Internship//vid//real"
fake_path = "C://Users//Sinchan A//Desktop//Internship//vid//fake"

X = []
y = []

def add_features(img_path, label):
    features = extract_features(img_path)
    if features is not None:
        X.append(features)
        y.append(label)

for img_name in os.listdir(real_path):
    img_path = os.path.join(real_path, img_name)
    add_features(img_path, 0)

for img_name in os.listdir(fake_path):
    img_path = os.path.join(fake_path, img_name)
    add_features(img_path, 1)

if X and y:
    X = np.array(X)
    y = np.array(y)

    # Split into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

    counter = Counter(y_train)
    print('Before SMOTE:', counter)

    # Oversampling the train dataset using SMOTE
    smt = SMOTE(random_state=42)
    X_train, y_train = smt.fit_resample(X_train, y_train)

    counter = Counter(y_train)
    print('After SMOTE:', counter)

    # Initialize and train the Naive Bayes model
    nb = GaussianNB()
    nb.fit(X_train, y_train)

    # Make predictions on the test set
    y_pred = nb.predict(X_test)

    # Evaluate the model
    accuracy = accuracy_score(y_test, y_pred)
    conf_matrix = confusion_matrix(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"Accuracy: {accuracy:.4f}")
    print("Confusion Matrix:")
    print(conf_matrix)
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")
else:
    print("No valid data was extracted from the images.")

Before SMOTE: Counter({0: 2749, 1: 2621})
After SMOTE: Counter({1: 2749, 0: 2749})
Accuracy: 0.5825
Confusion Matrix:
[[1071  808]
 [ 687 1015]]
Precision: 0.5568
Recall: 0.5964
F1-score: 0.5759


In [3]:
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

In [4]:
print(f'Precision: {precision}')
print(f'Recall: {recall}')
print(f'F1-score: {f1}')

Precision: 0.5567745474492595
Recall: 0.5963572267920094
F1-score: 0.5758865248226951
